# Lab 04: From LLM to Agent -- See the Difference

**Goal:** See the same task handled by a plain LLM vs an "agent" with tools.

**What you'll learn:**
- A plain LLM can only guess at answers requiring live data
- An "agent" (LLM + tools) can fetch real data and give accurate answers
- Tools bridge the gap between "knowing" and "doing"

**Scenarios:** Inventory lookup, compound interest, currency exchange, and multi-tool investment analysis

In [ ]:
import math
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model="llama3.2:1b")

## Scenario 1: "What is the stock level of Laptop Pro?"

### Plain LLM

In [ ]:
# Plain LLM -- can only guess at inventory levels
response = llm.invoke("What is the current stock level of the product 'Laptop Pro'? Give specific numbers.")
print(f"LLM says: {response.content}")

### Agent (LLM + Inventory Tool)

In [ ]:
def inventory_tool(product: str) -> str:
    """Returns current stock level for a product from the warehouse database."""
    inventory = {
        "Laptop Pro": "In Stock — 142 units",
        "Wireless Mouse": "Low Stock — 8 units",
        "USB-C Hub": "Out of Stock — 0 units",
    }
    return inventory.get(product, "Product not found")

stock_info = inventory_tool("Laptop Pro")
# Feed the tool result to the LLM
response = llm.invoke([
    SystemMessage(content="You are a helpful inventory assistant. Use the provided data to answer."),
    HumanMessage(content=f"Inventory data for 'Laptop Pro': {stock_info}\n\nWhat is the current stock level of Laptop Pro?"),
])
print(f"Agent says: {response.content}")
print(f"(Used inventory tool -> got: {stock_info})")

## Scenario 2: "Rs 10,000 at 8% compound interest for 5 years?"

### Plain LLM

In [ ]:
# Plain LLM -- struggles with compound interest formula
response = llm.invoke("If I invest Rs 10,000 at 8% annual compound interest for 5 years, how much will I have? Show the exact amount.")
print(f"LLM says: {response.content}")
print(f"Correct:  {10000 * (1 + 0.08) ** 5:.2f}")

### Agent (LLM + Finance Tool)

In [ ]:
def finance_tool(expression: str) -> str:
    """Safely evaluate a financial/math expression."""
    try:
        result = eval(expression, {"__builtins__": {}, "math": math})
        return f"{result:.2f}"
    except Exception as e:
        return f"Error: {e}"

calc_result = finance_tool("10000 * (1 + 0.08) ** 5")
response = llm.invoke([
    SystemMessage(content="You are a financial assistant. Use the calculation result to answer."),
    HumanMessage(content=f"Compound interest calculation: 10000 * (1 + 0.08)^5 = Rs {calc_result}\n\nIf I invest Rs 10,000 at 8% annual compound interest for 5 years, how much will I have?"),
])
print(f"Agent says: {response.content}")
print(f"(Used finance tool -> got: Rs {calc_result})")

## Scenario 3: "What is 5000 INR in USD?"

### Plain LLM

In [ ]:
# Plain LLM -- gives outdated or wrong exchange rate
response = llm.invoke("What is 5000 INR in USD? Give the exact amount using today's exchange rate.")
print(f"LLM says: {response.content}")

### Agent (LLM + Exchange Rate Tool)

In [ ]:
def exchange_rate_tool(pair: str) -> str:
    """Simulated forex API (in real life, this calls an exchange rate service)."""
    rates = {
        "INR-USD": "1 INR = 0.012 USD",
        "INR-EUR": "1 INR = 0.011 EUR",
        "USD-INR": "1 USD = 83.33 INR",
        "JPY-INR": "1 JPY = 0.56 INR",
    }
    return rates.get(pair, "Currency pair not available")

rate_info = exchange_rate_tool("INR-USD")
rate_value = rate_info.split("=")[1].strip().split()[0]  # extract "0.012"
usd_amount = finance_tool(f"5000 * {rate_value}")
response = llm.invoke([
    SystemMessage(content="You are a currency exchange assistant. Use the provided rate and calculation to answer."),
    HumanMessage(content=f"Current exchange rate: {rate_info}\nConverted amount: {usd_amount} USD\n\nWhat is 5000 INR in USD?"),
])
print(f"Agent says: {response.content}")
print(f"(Tool 1: exchange rate -> {rate_info})")
print(f"(Tool 2: finance -> {usd_amount} USD)")

## Scenario 4: The Full Picture -- Multi-Tool Investment Analysis

### Plain LLM

In [ ]:
question = "How much USD will I get if I invest Rs 10,000 at 8% compound interest for 5 years?"

print(f"Question: {question}\n")
print("--- Plain LLM ---")
response = llm.invoke(question)
print(f"  {response.content}")

### Agent (LLM + Finance Tool + Exchange Rate Tool)

In [ ]:
# Step 1: Calculate compound interest
future_value = finance_tool("10000 * (1 + 0.08) ** 5")
# Step 2: Get exchange rate
rate_info = exchange_rate_tool("INR-USD")
rate_value = rate_info.split("=")[1].strip().split()[0]  # extract "0.012"
# Step 3: Convert to USD
usd_value = finance_tool(f"{future_value} * {rate_value}")

response = llm.invoke([
    SystemMessage(content="Use the provided data to answer clearly."),
    HumanMessage(content=f"Investment of Rs 10,000 at 8% for 5 years grows to Rs {future_value}.\nExchange rate: {rate_info}\nConverted amount: ${usd_value} USD\n\n{question}"),
])
print(f"--- Agent ---")
print(f"  {response.content}")
print(f"  (Tool 1: finance -> Rs {future_value})")
print(f"  (Tool 2: exchange rate -> {rate_info})")
print(f"  (Tool 3: finance -> ${usd_value} USD)")

## TODO 1: Recipe Scaling Comparison

**Question:** "What ingredients do I need for Paneer Butter Masala for 8 people?"

**Steps:**
1. Ask the plain LLM (it will guess ingredients and quantities)
2. Create a `recipe_tool(dish)` that returns ingredients for 4 servings
3. Feed the tool result to the LLM and ask it to scale for 8 people
4. Compare the answers!

**Recipe data (for 4 servings):**
- "Paneer Butter Masala": "Paneer 250g, Butter 50g, Tomato Puree 200ml, Cream 100ml, Onion 2, Ginger-Garlic Paste 2 tbsp"
- "Dal Tadka": "Toor Dal 200g, Ghee 30g, Cumin Seeds 1 tsp, Onion 1, Tomato 2, Green Chili 2"
- "Vegetable Biryani": "Basmati Rice 300g, Mixed Vegetables 400g, Yogurt 100g, Biryani Masala 2 tbsp, Onion 3, Oil 60ml"

In [ ]:
# TODO: Implement recipe_tool and compare Plain LLM vs Agent for recipe scaling

## TODO 2: Trip Budget Multi-Tool

**Question:** "If a hotel in Tokyo costs 15,000 JPY per night for 5 nights, how much is that in INR?"

This needs TWO tools:
1. `finance_tool("15000 * 5")` -- calculates total cost in JPY
2. `exchange_rate_tool("JPY-INR")` -- gets JPY to INR rate (already defined above: 1 JPY = 0.56 INR)
3. `finance_tool()` -- converts total JPY to INR

Try building this multi-step agent!

In [ ]:
# TODO: Chain finance_tool and exchange_rate_tool to calculate trip budget

## Key Takeaways

- **Plain LLM:** Smart but blind (no access to live data)
- **Agent (LLM + Tools):** Smart AND connected to the world
- Tools give the LLM real information to work with
- The LLM's job is to **REASON**; tools provide the **DATA**
- This is the foundation of all agentic AI!